# Git 入门教程（PowerShell 版）

AGDR VLA 入门教程 · 第 1 章

**预计学习时长：1.5 小时**

> 本版本适用于 **Windows PowerShell**（Windows 10 / 11 自带）。
> macOS、Linux、Git Bash、WSL 的同学请打开同目录下的 bash 版。
> 演示代码使用 `%%script powershell`，在临时目录 `%TEMP%\git-demo` 中运行，不会影响你的真实项目。
> 如果你用的是 cmd，请先在 cmd 里输入 `powershell` 进入 PowerShell；本教程中的 `git` 命令在两者里写法相同。

## 如何使用

1. 先观看入门视频：<https://www.bilibili.com/video/BV1Hkr7YYEh8>
2. 按顺序阅读本 notebook 的 12 个小节，并运行其中的演示代码
3. 最后完成文末的三个任务

## 目录

1. Git 是什么，为什么需要版本控制
2. 安装与初始配置
3. 工作区、暂存区、本地仓库
4. 基本提交流程
5. 查看与撤销改动
6. 忽略文件
7. 分支
8. 合并与冲突
9. 远程仓库
10. 协作流程：Fork 与 Pull Request
11. 安全操作清单
12. 与 Agent 协作
13. 任务：作业一、二、三

## 01. Git 是什么，为什么需要版本控制

- **学习目标**：说清楚版本控制解决什么问题，区分 Git 与 GitHub

### 没有版本控制会怎样

你一定见过这样的文件名：`报告_final.docx`、`报告_final_v2.docx`、`报告_真的最终版.docx`。

问题在于：
- 不知道每个版本改了什么
- 想回到三天前的版本很困难
- 多人同时修改时，互相覆盖

**版本控制系统**就是用来解决这些问题的工具。

### Git 的核心想法

- **快照**：每次提交（commit）都会为整个项目保存一份快照，并记录作者、时间和说明
- **历史**：快照串成一条（或多条）历史，可以随时查看、对比、回到过去
- **分布式**：每个人电脑上都有完整的仓库和历史，不依赖某台服务器才能工作

### Git 与 GitHub 的区别

| | Git | GitHub |
| --- | --- | --- |
| 是什么 | 本地运行的版本控制工具 | 托管 Git 仓库的网站 / 平台 |
| 是否需要网络 | 不需要 | 需要 |
| 额外功能 | 无 | Fork、Pull Request、Issue 等协作功能 |

类似的平台还有 GitLab、Gitee。它们都基于 Git。

### 动手：确认 Git 已安装

In [ ]:
!git --version

如果提示找不到命令，请先阅读下一节《安装与初始配置》。

## 02. 安装与初始配置

- **学习目标**：安装 Git，并配置提交时使用的用户名和邮箱

### 安装

| 系统 | 方法 |
| --- | --- |
| Windows | 从 <https://git-scm.com/download/win> 下载安装，或在 PowerShell 运行 `winget install --id Git.Git`；安装后 PowerShell、cmd 都可以直接使用 `git` |
| macOS | 终端运行 `xcode-select --install` 或 `brew install git` |
| Linux（Debian/Ubuntu） | `sudo apt install git` |

安装后重新打开终端，运行下面的单元格确认。

In [ ]:
!git --version

### 配置用户信息

每次提交都会记录作者信息，所以第一次使用前要配置姓名和邮箱。

请在**终端**里运行（把引号里的内容换成你自己的）：

```bash
git config --global user.name "你的名字"
git config --global user.email "你的邮箱"
```

- `--global` 表示对当前用户的所有仓库生效
- 邮箱建议与你的 GitHub 账号邮箱一致，这样提交才会关联到你的账号

> 这两条命令会修改你的真实配置，所以没有放在可运行的单元格里。

### 查看当前配置

In [ ]:
!git config --global --list

## 03. 工作区、暂存区、本地仓库

- **学习目标**：理解文件在三个区域之间的流动，会读 `git status`

### 三个区域

```
工作区            暂存区               本地仓库
(Working Dir)     (Staging / Index)    (Repository)
     │   git add      │    git commit      │
     │ ─────────────▶ │ ─────────────────▶ │
     │                │                    │
```

- **工作区**：你正在编辑的文件
- **暂存区**：下一次提交准备包含的内容，相当于"待提交清单"
- **本地仓库**：已经提交、永久记录在历史里的快照

暂存区的意义：你可以只挑选一部分改动来提交，让每次提交只做一件事。

### 演示：观察文件状态的变化

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
git config user.name demo
git config user.email demo@example.com
Set-Content -Path a.txt -Value "hello"
git status

`a.txt` 显示为 **Untracked**（未跟踪）：Git 看到了它，但还没有管它。

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
git add a.txt
git status

文件进入暂存区，显示为 **Changes to be committed**。

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
git commit -m "add a.txt"
git status

提交之后，`git status` 显示 *working tree clean*，说明工作区和最近一次提交一致。

## 04. 基本提交流程

- **学习目标**：掌握 `git add`、`git commit`、`git log`，并写出清晰的提交信息

### 流程总览

```bash
git init            # 创建仓库（每个项目只做一次）
git add <文件>      # 放入暂存区
git commit -m "说明"  # 生成一次提交
git log             # 查看历史
```

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
git config user.name demo
git config user.email demo@example.com
Set-Content -Path main.py -Value "print('hello')"
git add main.py
git commit -m "feat: add main.py"

再做一次修改并提交，让历史多一条记录：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
Add-Content -Path main.py -Value "print('world')"
git add main.py
git commit -m "feat: print world"
git log --oneline

### 常用写法

- `git add .`：暂存当前目录下所有改动（使用前先用 `git status` 看一眼，避免带入不该提交的文件）
- `git commit -m "..."`：直接在命令里写提交信息
- `git log --oneline`：每次提交一行，方便浏览
- `git log -p`：同时显示每次提交的具体改动

### 怎样写好提交信息

- 一句话说明"做了什么"，而不是"改了点东西"
- 一次提交只做一件事
- 常见前缀：`feat:` 新功能、`fix:` 修复、`docs:` 文档

| 不好 | 好 |
| --- | --- |
| `update` | `fix: 修复冒泡排序对空列表报错` |
| `改了一些东西` | `docs: 补充 Git 安装说明` |

## 05. 查看与撤销改动

- **学习目标**：用 `git diff` 查看改动，用 `git restore` 安全地撤销

### 查看改动

| 命令 | 对比的内容 |
| --- | --- |
| `git diff` | 工作区 与 暂存区 |
| `git diff --staged` | 暂存区 与 最近一次提交 |

### 撤销改动

| 目的 | 命令 |
| --- | --- |
| 丢弃工作区的修改，恢复成暂存区 / 最近提交的样子 | `git restore <文件>` |
| 把文件从暂存区拿出来（改动保留在工作区） | `git restore --staged <文件>` |

> ⚠️ `git restore <文件>` 会**丢弃**你尚未提交的修改，且无法找回。执行前先用 `git diff` 确认。

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
git config user.name demo
git config user.email demo@example.com
Set-Content -Path note.txt -Value "v1"
git add note.txt
git commit -m "add note"
Set-Content -Path note.txt -Value "v2"
git diff

`-v1 / +v2` 表示这一行从 `v1` 变成了 `v2`。现在把它暂存，再取消暂存：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
git add note.txt
git diff --staged
git restore --staged note.txt
git status --short

最后丢弃这次修改，文件恢复为 `v1`：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
git restore note.txt
Get-Content note.txt

## 06. 忽略文件：.gitignore

- **学习目标**：会用 `.gitignore` 排除不该提交的文件

### 为什么要忽略

有些文件不应该进入版本库：

- 密钥、令牌、`.env`：提交后可能泄露
- 数据集、模型权重：体积大，会拖垮仓库
- `__pycache__/`、`.ipynb_checkpoints/`：自动生成，无意义
- 系统文件：`.DS_Store`、`Thumbs.db`

### 写法

在仓库根目录创建 `.gitignore`，每行一条规则：

```gitignore
# 注释
__pycache__/
.ipynb_checkpoints/
.env
*.pt
data/
```

- `*.pt`：忽略所有 `.pt` 文件
- `data/`：忽略整个 `data` 目录

> 注意：`.gitignore` 只对**还没被跟踪**的文件有效。已经提交过的文件，需要先从 Git 中移除跟踪。
> 一旦密钥被提交并推送过，就应当视为已泄露，需要立即更换。

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
Set-Content -Path .env -Value "SECRET=123"
Set-Content -Path app.py -Value "print(1)"
git status --short

`.env` 和 `app.py` 都被列为未跟踪。写入 `.gitignore` 后再看：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
Set-Content -Path .gitignore -Value ".env"
git status --short

`.env` 不见了，只剩 `.gitignore` 和 `app.py`。

## 07. 分支

- **学习目标**：会创建、切换分支，理解“每个任务一个分支”的做法

### 分支是什么

分支是指向某次提交的一个"可移动的标签"。创建分支几乎没有成本，所以可以放心地为每个任务单独开一个分支，互不干扰。

```
main:      A ── B ── C
                     \
feature:              D ── E
```

### 常用命令

| 命令 | 作用 |
| --- | --- |
| `git branch` | 列出分支，`*` 表示当前分支 |
| `git switch -c <名称>` | 创建并切换到新分支 |
| `git switch <名称>` | 切换到已有分支 |

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
git config user.name demo
git config user.email demo@example.com
Set-Content -Path f.txt -Value "base"
git add f.txt
git commit -m "base"
git switch -c feature
Add-Content -Path f.txt -Value "feature work"
git commit -am "work on feature"
git branch

切回 `main`，观察文件内容：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
git switch main
Get-Content f.txt
Write-Output "---"
git switch feature
Get-Content f.txt

同一个文件在两个分支里内容不同，这就是分支的隔离作用。

### 命名建议

- `feature/xxx`：新功能
- `fix/xxx`：修 bug
- `homework/xxx`：本教程的作业分支

## 08. 合并与冲突

- **学习目标**：会用 `git merge`，看懂冲突标记并手动解决一次冲突

### 合并

把一个分支的改动并入当前分支：

```bash
git switch main
git merge feature
```

如果两个分支改的是**不同位置**，Git 会自动合并。如果改的是**同一处**，就会产生**冲突**，需要人来决定保留什么。

### 演示：制造一次冲突

In [ ]:
%%script powershell -NoProfile -Command -
$demo = "$env:TEMP\git-demo"
if (Test-Path $demo) { Remove-Item -Recurse -Force $demo }
New-Item -ItemType Directory -Path $demo | Out-Null
Set-Location $demo
git init -b main
git config core.autocrlf false
git config user.name demo
git config user.email demo@example.com
Set-Content -Path cfg.txt -Value "color=red"
git add cfg.txt
git commit -m "base"
git switch -c feature
Set-Content -Path cfg.txt -Value "color=blue"
git commit -am "feature: blue"
git switch main
Set-Content -Path cfg.txt -Value "color=green"
git commit -am "main: green"
git merge feature

Git 报告冲突并停下来等你处理。看看文件里写了什么：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
Get-Content cfg.txt

```
<<<<<<< HEAD
color=green
=======
color=blue
>>>>>>> feature
```

- `<<<<<<< HEAD` 到 `=======`：当前分支（main）的内容
- `=======` 到 `>>>>>>> feature`：要合并进来的分支的内容

### 解决步骤

1. 编辑文件，决定最终内容，并**删除所有标记行**
2. `git add <文件>` 标记为已解决
3. `git commit` 完成合并

这里我们选择保留 `color=blue`：

In [ ]:
%%script powershell -NoProfile -Command -
Set-Location "$env:TEMP\git-demo"
Set-Content -Path cfg.txt -Value "color=blue"
git add cfg.txt
git commit -m "merge feature, keep blue"
git log --oneline --graph

> 如果合并到一半想放弃，可以运行 `git merge --abort` 回到合并之前。

## 09. 远程仓库

- **学习目标**：区分 clone / fetch / pull / push，了解认证方式

### 远程仓库

远程仓库是托管在服务器（如 GitHub）上的仓库副本，用来备份和协作。默认的远程名叫 `origin`。

### 常用命令

| 命令 | 作用 |
| --- | --- |
| `git clone <地址>` | 把远程仓库完整复制到本地 |
| `git remote -v` | 查看远程地址 |
| `git fetch` | 下载远程的新提交，**不**修改你的工作区 |
| `git pull` | `fetch` + 合并到当前分支 |
| `git push` | 把本地提交上传到远程 |

`fetch` 比较安全，只是"看看远程有什么新东西"；`pull` 会真的改动你的分支。

### 认证方式

| 方式 | 说明 |
| --- | --- |
| HTTPS | 用账号 + 个人访问令牌（Personal Access Token）；GitHub 已不支持账号密码 |
| SSH | 本地生成密钥对，把公钥添加到 GitHub，之后免输入 |

SSH 简要步骤：

```bash
ssh-keygen -t ed25519 -C "你的邮箱"
ssh -T git@github.com         # 添加公钥后，用它测试连接
```

查看公钥内容（复制到 GitHub → Settings → SSH keys），不同终端的命令不同：

| 终端 | 命令 |
| --- | --- |
| PowerShell | `Get-Content $HOME\.ssh\id_ed25519.pub` |
| cmd | `type %USERPROFILE%\.ssh\id_ed25519.pub` |
| Git Bash / macOS / Linux | `cat ~/.ssh/id_ed25519.pub` |

> 私钥（没有 `.pub` 后缀的那个文件）绝对不要发给别人，也不要提交到仓库。

### 查看当前仓库的远程

In [ ]:
!git remote -v

如果上面没有输出，说明当前目录不是克隆来的仓库或没有配置远程，这是正常的。

## 10. 协作流程：Fork 与 Pull Request

- **学习目标**：能完整走通一次“Fork → 修改 → Pull Request”

### 为什么要 Fork

你通常没有权限直接修改别人的仓库。**Fork** 会把仓库复制一份到你自己的账号下，你可以随意修改，再请求原作者合并。

### 完整流程

```
原仓库 (upstream)  ──Fork──▶  你的 Fork (origin)  ──clone──▶  本地
        ▲                                                       │
        └────────────── Pull Request ◀──── push ◀── commit ◀────┘
```

1. 在 GitHub 上点击 **Fork**
2. `git clone` 你**自己的** Fork 到本地
3. `git switch -c homework/xxx` 新建分支
4. 修改、`git add`、`git commit`
5. `git push -u origin homework/xxx`
6. 在 GitHub 页面点击 **Compare & pull request**，填写标题和说明，提交

### 保持与原仓库同步

```bash
git remote add upstream <原仓库地址>
git fetch upstream
git merge upstream/main
```

### Pull Request 写什么

- **标题**：简短说明做了什么
- **说明**：为什么改、怎么验证、有没有需要 reviewer 注意的地方

### 作为 reviewer

- 先读改动（Files changed），再提意见
- 对事不对人，具体指出哪一行、建议怎么改

## 11. 安全操作清单

- **学习目标**：知道哪些 Git 命令会丢数据，执行前先做什么

### 会丢数据的命令

| 命令 | 风险 |
| --- | --- |
| `git restore <文件>` | 丢弃未提交的修改，无法找回 |
| `git reset --hard` | 丢弃所有未提交的修改，并移动分支位置 |
| `git clean -fd` | 删除所有未跟踪的文件和目录 |
| `git push --force` | 覆盖远程历史，可能抹掉别人的提交 |
| `git branch -D <分支>` | 强制删除分支，未合并的提交可能丢失 |

### 执行前的检查习惯

1. `git status`：当前在哪个分支，有哪些未提交的改动
2. `git diff`：确认要丢弃的是什么
3. 不确定时，先建一个备份分支：`git branch backup-xxx`
4. 强制推送前，确认没有别人在这个分支上工作；能用 `--force-with-lease` 就不用 `--force`

### 提交过的内容通常能找回

只要内容**提交过**，即使分支被移动，也可以通过 `git reflog` 找到之前的位置：

```bash
git reflog
git switch -c rescue <提交编号>
```

而**从未提交过**的修改，Git 无能为力。所以：勤提交。

## 12. 与 Agent 协作使用 Git

- **学习目标**：让 Agent 辅助 Git 操作的同时，保持对仓库的控制

### Agent 能帮你做什么

- 解释某条命令、某段报错是什么意思
- 总结 `git diff` / `git log` 的内容
- 根据改动草拟提交信息
- 帮你排查冲突为什么出现

### 使用原则

1. **先理解，再执行**：Agent 给出的命令，你要能说清楚每一部分的含义
2. **不让 Agent 擅自提交或推送**：提交和推送由你决定，在提示里明确写出
3. **危险命令要二次确认**：遇到 `reset --hard`、`clean`、`push --force`，先问清楚后果
4. **执行后检查**：用 `git status`、`git log`、`git diff` 自己确认结果
5. **敏感信息不外传**：不要把密钥、令牌、`.env` 内容发给 Agent

### 提示词示例

```text
请解释下面这段 git diff，不要修改任何文件：
<粘贴内容>
```

```text
根据 git diff --staged 给我 3 个提交信息候选，不要执行 commit。
```

```text
我在 merge 时出现冲突，请先说明冲突原因和可选方案，等我确认后再改文件。
```

### 一个好习惯

每次让 Agent 动过仓库之后，运行：

In [ ]:
!git status --short
!git log --oneline -5

## 13. 任务

完成以上学习后，请完成下面三个任务。

### 作业一：Fork 并提交冒泡排序

1. 将本仓库 Fork 到你自己的 GitHub 账号下。
2. 在你的 Fork 中新建分支，例如 `homework/bubble-sort-你的名字`。
3. 在 `1. Git的使用与操作/submission/` 目录下新建一个 Python 文件，文件名为 `bubble_sort_你的GitHub用户名.py`。
4. 文件中实现冒泡排序函数 `bubble_sort(arr)`，返回排序后的列表（升序）。
5. 提交（commit）并推送（push）到你的 Fork。
6. 在原仓库发起 Pull Request，标题格式为 `[作业一] 你的名字`。

检查标准：

- 函数名与参数符合要求，能处理空列表和已排序列表
- 不修改传入的原列表（可选加分项）
- Pull Request 标题格式正确，分支与文件名符合规则

函数写法参考（请自己动手写，不要直接复制）：

```python
def bubble_sort(arr):
    arr = list(arr)
    n = len(arr)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


if __name__ == "__main__":
    print(bubble_sort([5, 2, 9, 1]))
```

### 作业二：Agent 知识点选择题

使用推荐的 Agent 工具（Codex、Claude Code、DeepSeek Harness 任选其一）完成。

1. 在仓库**根目录**启动 Agent，直接开始对话即可，不需要输入特殊提示词。
2. 按 Agent 的引导选择「答题模式」和「第 1 章」，然后按提示作答，直到它宣布测验完成。
3. 完成后，Agent 会给出一份总结，把它复制到作业三的 Issue 里。

注意：

- 请自己作答，不要让 Agent 直接给出答案
- 答错的题目要弄懂原因再继续
- 如果 Agent 出的题目明显有错误，记录下来，放进作业三的 Issue

### 作业三：提交一个 Issue

在原仓库新建一个 Issue，内容包括：

- **标题**：`[作业三] 你的名字`
- **正文**：
  1. 作业二的最终结论与最后一组题目（可以简要写）
  2. 本章中你遇到的一个问题、不清楚的地方，或对教程的一条改进建议
  3. 如果发现 Agent 出题或讲解有错误，写出具体题目和你的判断

Issue 不需要很长，写清楚问题即可。

### 完成检查表

- [ ] 观看入门视频
- [ ] 作业一：Pull Request 已提交
- [ ] 作业二：Agent 判定完成，结论已记录
- [ ] 作业三：Issue 已提交